# 01 - Análise Exploratória

Este notebook abre a trilha analítica do projeto WineQT. A ideia aqui é transformar dados brutos em contexto de decisão: entender a estrutura do conjunto, validar a regra da variável alvo e identificar sinais que expliquem o comportamento da classe de alta qualidade.

Ao final desta etapa, queremos responder três perguntas práticas:
- o conjunto está consistente para modelagem?
- a classe positiva (`quality >= 7`) é desbalanceada?
- há padrões (correlação, distribuição e outliers) que impactam a escolha de métricas e modelos?


In [ ]:
from pathlib import Path

import pandas as pd

data_path = Path("../data/WineQT.csv")
df = pd.read_csv(data_path)

print(f"Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")
print("\nTipos de dados:")
print(df.dtypes.to_string())
print("\nValores ausentes por coluna:")
missing = df.isna().sum()
print(missing[missing > 0].to_string())
print(f"\nDuplicatas: {df.duplicated().sum()}")

## Definição da variável alvo

A regra do enunciado foi aplicada para transformar a qualidade em um problema binário:

- alta qualidade: `quality >= 7`
- baixa ou média qualidade: `quality < 7`

Essa decisão foi usada de forma consistente em toda a análise e na comparação entre modelos.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# regra definida para o desafio: alta qualidade = quality >= 7
df["target"] = (df["quality"] >= 7).astype(int)
target_counts = df["target"].value_counts().sort_index()

print("Contagem por classe:")
print(target_counts.to_string())
print(f"\nProporção da classe positiva: {target_counts.get(1, 0) / len(df):.2%}")

plt.figure(figsize=(6, 4))
sns.countplot(
    data=df,
    x="target",
    hue="target",
    palette="Set2",
    legend=False,
)
plt.title(
    "Distribuição da classe alvo", fontsize=13, fontweight="bold", loc="center", pad=12
)
plt.xlabel("Classe (0 = baixa/média, 1 = alta)")
plt.ylabel("Contagem")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(df["quality"], bins=10, kde=True, color="steelblue")
plt.title(
    "Distribuição da qualidade do vinho",
    fontsize=13,
    fontweight="bold",
    loc="center",
    pad=12,
)
plt.xlabel("Qualidade")
plt.ylabel("Frequência")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
ax = plt.gca()
ax.pie(
    target_counts.values,
    labels=["Baixa/Média", "Alta"],
    autopct="%1.1f%%",
    startangle=90,
    colors=["#9ecae1", "#fd8d3c"],
)
ax.set_title(
    "Proporção da classe positiva",
    fontsize=13,
    fontweight="bold",
    loc="center",
    pad=14,
)
plt.tight_layout()
plt.show()

## Correlação, distribuição e sinais de decisão

A partir da matriz de correlação e da comparação por classe, é possível identificar quais variáveis têm relação mais forte com a qualidade e com o comportamento do conjunto. Esse passo ajuda a justificar o uso de métricas adequadas para dados desbalanceados e a entender por que certos modelos melhoram a classificação da classe positiva.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

corr = df.drop(columns=["Id", "quality", "target"]).corr(numeric_only=True)

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=False, cmap="coolwarm", fmt=".2f")
plt.title(
    "Matriz de correlação das variáveis preditivas",
    fontsize=14,
    fontweight="bold",
    loc="center",
)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="target", y="alcohol", hue="target", palette="Set2", legend=False)
plt.title(
    "Distribuição de álcool por classe", fontsize=13, fontweight="bold", loc="center"
)
plt.xlabel("Classe")
plt.ylabel("Alcohol")
plt.xticks([0, 1], ["Baixa/Média", "Alta"])
plt.tight_layout()
plt.show()

missing_plot = df.isna().sum().sort_values(ascending=False)
missing_plot = missing_plot[missing_plot > 0]

if not missing_plot.empty:
    plt.figure(figsize=(8, 4))
    sns.barplot(x=missing_plot.index, y=missing_plot.values, hue=missing_plot.index, palette="crest", legend=False)
    plt.title(
        "Valores ausentes por variável", fontsize=13, fontweight="bold", loc="center"
    )
    plt.xticks(rotation=45)
    plt.ylabel("Quantidade")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum valor ausente encontrado no conjunto.")

## Outliers das variáveis físico-químicas

Como parte do EDA, avaliamos possíveis outliers com boxplots e também quantificamos a proporção de pontos fora dos limites do método IQR (Q1 - 1.5*IQR e Q3 + 1.5*IQR).

Nesta etapa, o objetivo é diagnóstico. A remoção ou tratamento deve ser validada depois na modelagem para evitar perda de sinal relevante da classe positiva.


In [ ]:
colunas = [
    "fixed acidity",
    "volatile acidity",
    "citric acid",
    "residual sugar",
    "chlorides",
    "free sulfur dioxide",
    "total sulfur dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
]

ncols = 3
nrows = (len(colunas) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4.7 * nrows))
axes = axes.flatten()

for i, coluna in enumerate(colunas):
    sns.boxplot(
        data=df,
        y=coluna,
        ax=axes[i],
        color="steelblue",
        width=0.4,
        flierprops={"marker": "o", "markersize": 3, "alpha": 0.5},
    )
    axes[i].set_title(coluna, fontsize=13, fontweight="bold", loc="center")
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Valor")

for ax in axes[len(colunas) :]:
    ax.set_visible(False)

fig.suptitle(
    "Distribuição e possíveis outliers das variáveis físico-químicas",
    fontsize=13,
    fontweight="bold",
    y=0.988,
)
fig.subplots_adjust(top=0.93, hspace=0.32, wspace=0.2)
plt.show()

outlier_summary = []
for coluna in colunas:
    q1 = df[coluna].quantile(0.25)
    q3 = df[coluna].quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr

    mask_outlier = (df[coluna] < lim_inf) | (df[coluna] > lim_sup)
    qtd_outliers = int(mask_outlier.sum())
    perc_outliers = 100 * qtd_outliers / len(df)

    outlier_summary.append(
        {
            "variavel": coluna,
            "limite_inferior": lim_inf,
            "limite_superior": lim_sup,
            "qtd_outliers": qtd_outliers,
            "perc_outliers": perc_outliers,
        }
    )

outlier_df = pd.DataFrame(outlier_summary).sort_values(
    by="perc_outliers", ascending=False
)
print("Resumo de outliers (critério IQR):")
display(outlier_df.style.format({"perc_outliers": "{:.2f}%"}))

## Interpretação do EDA

A leitura exploratória mostra um cenário claro: a classe de alta qualidade é minoritária, então a avaliação do projeto precisa priorizar **F1**, **recall** e métricas probabilísticas, em vez de depender apenas de acurácia.

Também observamos que `alcohol` diferencia bem as classes e que a matriz de correlação sugere efeito combinado entre variáveis, não uma relação simples de causa única. Na prática, isso favorece modelos mais flexíveis e avaliação cuidadosa de generalização.

Por fim, o diagnóstico de outliers por boxplot e IQR indica caudas relevantes (como em sulfur dioxide, chlorides e residual sugar). A decisão metodológica foi manter esses pontos no EDA e validar tratamento apenas em etapa experimental, para evitar perda prematura de sinal da classe positiva.
